# PFAI Semester Project: Web Scraping (Member 1)

**Course:** Programming for AI (Instructor: Mahhek Tahir)
**Data source:** [Books to Scrape](https://books.toscrape.com/), a practice website made for learning web scraping.

### Objective
Collect every book listing on the website, visit each book's own page to get the extra attributes, and save the data as a structured CSV file with **one book per row**. The other parts of the project (data understanding, cleaning, EDA and the machine-learning models) all start from the files this notebook produces.

### Plan
1. **Setup:** libraries, constants and one shared HTTP session
2. **Fetch helper:** download a page safely, with retries
3. **Inspect the site structure:** total results, pages and categories
4. **Collect book links:** follow the pagination through every catalogue page
5. **Parse one book page:** extract every attribute as raw text
6. **Scrape all books:** visit all product pages, politely
7. **Raw dataset:** save the text exactly as scraped (`books_raw.csv`)
8. **Structured dataset:** convert text to proper types (`books_structured.csv`)
9. **Data-quality report:** unique records, duplicates, missing, constant and inconsistent fields
10. **Summary and data dictionary**

### Responsible scraping
The site is meant for practice, but we still keep the load low: a small number of parallel requests (5), a short pause after each request, no retries on pages that do not exist (404), and a cached copy of the raw data so that re-running the notebook does **not** download all 1,000 pages again.

## 1. Setup

In [1]:
# Run once if the libraries are not installed:
# !pip install requests beautifulsoup4 pandas

import os
import re
import time
import logging
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/"
START_URL = urljoin(BASE_URL, "catalogue/page-1.html")

RAW_FILE = "books_raw.csv"                # text exactly as it appears on the site
STRUCTURED_FILE = "books_structured.csv"  # cleaned types, ready for analysis

MAX_WORKERS = 5          # parallel requests for the product pages (kept small to be polite)
DELAY = 0.2              # seconds to pause after every request
FORCE_RESCRAPE = False   # True = ignore the saved raw CSV and download everything again

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s",
                    datefmt="%H:%M:%S", force=True)   # force=True so logs show in Jupyter
log = logging.getLogger(__name__)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (PFAI student project)"})
print("Setup complete")

Setup complete


**What was done:** Imported the libraries, defined the constants used in the rest of the notebook and created one shared `requests.Session`.

**Why:**
- `requests` downloads web pages, `BeautifulSoup` reads the HTML, `pandas` builds and saves the table, `re` pulls numbers out of text, and `ThreadPoolExecutor` lets us download a few pages at the same time.
- Keeping all settings (URLs, file names, number of workers, delay) at the top means they can be changed in one place.
- A `Session` reuses the same network connection for every request, which is faster and puts less load on the server than opening a new connection each time. The `User-Agent` header identifies our scraper honestly.
- `RATING_MAP` is needed because the site stores the star rating as a word (`"Three"`), not as a number.

**Output:** Only the message *Setup complete*; this cell prepares the environment and produces no data.

## 2. Fetch helper

In [2]:
def get_soup(url, retries=3):
    """Download a page and return it as a BeautifulSoup object (None if it fails)."""
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, timeout=15)
            if r.status_code == 404:              # page does not exist: retrying will not help
                log.error("404 Not Found: %s", url)
                return None
            r.raise_for_status()
            r.encoding = "utf-8"                  # the site does not declare it, otherwise '£' becomes 'Â£'
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            log.warning("Attempt %d failed for %s: %s", attempt, url, e)
            time.sleep(2 * attempt)               # wait longer after each failure (2 s, 4 s, 6 s)
    log.error("Giving up on %s", url)
    return None

test = get_soup(START_URL)
print("Page title:", test.title.get_text(strip=True))

Page title: All products | Books to Scrape - Sandbox


**What was done:** Wrote `get_soup(url)`, which downloads one page and returns it as a BeautifulSoup object that can be searched with CSS selectors. It was tested on the first catalogue page.

**Why:** Every later step downloads pages, so the error handling lives in one function:
- `timeout=15` stops the notebook from hanging forever on a slow response.
- `raise_for_status()` turns server errors (e.g. 500) into exceptions so they are caught and **retried up to 3 times**, waiting a bit longer each time (2 s, 4 s, 6 s).
- A **404** means the page does not exist, so we stop immediately instead of sending useless retries.
- The site does not declare its character encoding, so `requests` guesses wrong and the pound sign appears as `Â£`. Setting `r.encoding = "utf-8"` fixes this.
- If a page still fails, the function returns `None` instead of crashing, so one bad page cannot stop the whole scrape.

**Output:** The title of the first catalogue page, which confirms that the website is reachable and the HTML is being parsed.

## 3. Inspecting the website structure
Before scraping, we check how the site is organised: how many books it reports, how many listing pages there are, how many books each page shows, and which categories exist.

In [3]:
first_page = get_soup(START_URL)

results_text = first_page.select_one("form.form-horizontal").get_text(" ", strip=True)
pages_text = first_page.select_one("li.current").get_text(strip=True)
books_on_page = len(first_page.select("article.product_pod"))
site_categories = [a.get_text(strip=True)
                   for a in first_page.select("div.side_categories ul li ul li a")]

print("Results text :", results_text)
print("Pagination   :", pages_text)
print("Books on page:", books_on_page)
print("Categories in sidebar:", len(site_categories))
print(site_categories[:10], "...")

Results text : 1000 results - showing 1 to 20 .
Pagination   : Page 1 of 50
Books on page: 20
Categories in sidebar: 50
['Travel', 'Mystery', 'Historical Fiction', 'Sequential Art', 'Classics', 'Philosophy', 'Romance', 'Womens Fiction', 'Fiction', 'Childrens'] ...


**What was done:** Downloaded the first catalogue page and read four things from it: the results counter, the pagination label, the number of book cards on the page and the category list in the left sidebar.

**Why:** The project asks us to inspect the site structure and work out how listing pages, product links, pagination and product pages can be reached. Knowing the expected totals in advance also lets us check later that nothing was missed.

**What the output shows / observations:**
<!-- VERIFY: check these numbers against your own output -->
- The site reports **1000 results**, shown **20 per page** over **50 pages** ("Page 1 of 50").
- The sidebar lists **50 categories**.
- Each book card (`article.product_pod`) only shows the title, price, rating and stock status. The UPC, category, description, tax and stock count are only on the book's own page, so **each product page has to be visited**.
- Every listing page has a **next** button (`li.next a`) linking to the following page, so we can move through all pages by following it until it disappears.
- Each book's page shows its category in the breadcrumb (Home › Books › *Category* › *Title*). Because of this, we do not need to scrape the 50 category pages separately; the sidebar list is kept only to cross-check our categories in Section 9.

## 4. Collecting the book links (pagination)

In [4]:
def get_all_book_urls(first_soup=None):
    """Follow the 'next' button through every catalogue page and collect
    the absolute URL of each book's product page."""
    book_urls = []
    url, soup, page = START_URL, first_soup, 1
    while url:
        if soup is None:                       # page 1 is already downloaded, so it is reused
            soup = get_soup(url)
            time.sleep(DELAY)
        if soup is None:
            break
        for article in soup.select("article.product_pod"):
            href = article.h3.a["href"]            # relative link, e.g. 'a-light-in-the-attic_1000/index.html'
            book_urls.append(urljoin(url, href))   # turn it into a full URL
        if page % 10 == 0:
            log.info("Page %d: %d book links so far", page, len(book_urls))
        next_btn = soup.select_one("li.next a")
        url = urljoin(url, next_btn["href"]) if next_btn else None
        soup, page = None, page + 1
    return book_urls

book_urls = get_all_book_urls(first_page)
print(len(book_urls), "book links found")
print(len(set(book_urls)), "unique links")
book_urls[:3]

00:38:32  Page 10: 200 book links so far
00:38:38  Page 20: 400 book links so far
00:38:44  Page 30: 600 book links so far
00:38:51  Page 40: 800 book links so far
00:38:56  Page 50: 1000 book links so far


1000 book links found
1000 unique links


['https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html',
 'https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html',
 'https://books.toscrape.com/catalogue/soumission_998/index.html']

**What was done:** Starting from page 1, the function reads every book card on a page, stores the link to that book's page, then follows the **next** button to the following page. It stops when there is no next button (the last page).

**Why:**
- Following the next button is more reliable than assuming there are exactly 50 pages: if the number of books changed, the code would still visit every page.
- The links on the page are *relative* (e.g. `a-light-in-the-attic_1000/index.html`); `urljoin` combines them with the current page's address to make full URLs that can be downloaded.
- Page 1 was already downloaded in Section 3, so it is reused instead of being requested twice.
- We print both the total and the number of unique links to make sure no book was collected twice.

**What the output shows / observations:**
<!-- VERIFY -->
- **1000 links** were found and **all 1000 are unique**, which matches the 1000 results reported by the site. No book was missed or counted twice during pagination.
- The first three links are shown to confirm they are complete, working URLs.

## 5. Parsing a single book page
`parse_book` visits one product page and extracts every attribute **as the raw text shown on the site**. Converting text to numbers is done later, in Section 8, so the raw data is preserved exactly as it was collected.

In [5]:
def parse_book(url):
    """Visit one product page and return its attributes as raw text."""
    soup = get_soup(url)
    if soup is None:
        return None
    main = soup.select_one("div.product_main")

    # 'Product Information' table -> {'UPC': '...', 'Price (excl. tax)': '£51.77', ...}
    table = {row.th.get_text(strip=True): row.td.get_text(strip=True)
             for row in soup.select("table.table-striped tr")}

    # Breadcrumb: Home > Books > <Category> > <Title>
    crumbs = soup.select("ul.breadcrumb li")
    category = crumbs[2].get_text(strip=True) if len(crumbs) > 2 else None

    # The rating is stored in the class name: <p class="star-rating Three">
    rating_tag = main.select_one("p.star-rating")
    rating_word = next((c for c in rating_tag["class"] if c in RATING_MAP), None) if rating_tag else None

    # The description is the <p> right after the 'Product Description' header (some books have none)
    desc_header = soup.select_one("#product_description")
    desc_p = desc_header.find_next_sibling("p") if desc_header else None

    img = soup.select_one("#product_gallery img")

    return {
        "title": main.h1.get_text(strip=True),
        "category": category,
        "price_text": main.select_one("p.price_color").get_text(strip=True),
        "price_excl_tax_text": table.get("Price (excl. tax)"),
        "price_incl_tax_text": table.get("Price (incl. tax)"),
        "tax_text": table.get("Tax"),
        "rating_text": rating_word,
        "availability_text": table.get("Availability"),
        "num_reviews_text": table.get("Number of reviews"),
        "upc": table.get("UPC"),
        "product_type": table.get("Product Type"),
        "description": desc_p.get_text(strip=True) if desc_p else None,
        "image_url": urljoin(url, img["src"]) if img else None,
        "url": url,
    }

parse_book(book_urls[0])     # test on one book before scraping all of them

{'title': 'A Light in the Attic',
 'category': 'Poetry',
 'price_text': '£51.77',
 'price_excl_tax_text': '£51.77',
 'price_incl_tax_text': '£51.77',
 'tax_text': '£0.00',
 'rating_text': 'Three',
 'availability_text': 'In stock (22 available)',
 'num_reviews_text': '0',
 'upc': 'a897fe39b1053632',
 'product_type': 'Books',
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced a

**What was done:** Wrote `parse_book(url)` and tested it on the first book. It reads:

| Attribute | Where it is on the page |
|---|---|
| Title | the `<h1>` inside `div.product_main` |
| Category | 3rd item of the breadcrumb trail |
| Price | `p.price_color` |
| UPC, product type, prices excl./incl. tax, tax, availability, number of reviews | rows of the *Product Information* table, read into a dictionary |
| Star rating | the class name of `p.star-rating` (e.g. `star-rating Three`) |
| Description | the `<p>` directly after the `#product_description` header |
| Image URL, product URL | the gallery `<img>` and the page address |

**Why:**
- The product table is read into a dictionary keyed by the row header. Looking values up by name (`table.get("UPC")`) instead of by position keeps the code working even if the rows were in a different order, and `.get()` returns `None` instead of crashing if a row is missing.
- Some books have **no description**; the `if desc_header else None` check records these as missing instead of causing an error.
- Values are kept as the **original text** (e.g. `£51.77`, `In stock (22 available)`, `Three`) so we have a true *raw* dataset, as the deliverables require. Turning them into numbers is a separate, documented step.
- Testing on one book first lets us catch selector mistakes before sending 1,000 requests.

**What the output shows / observations:**
<!-- VERIFY -->
- The test record has all 14 fields filled for *A Light in the Attic*.
- The three price fields show the same value and the tax is `£0.00`, an early sign that the tax fields add no information.
- The description ends with the text `...more`, which is part of the website's layout, not the book's description. It is removed in Section 8.

## 6. Scraping all books

In [6]:
failed = []

def safe_parse(url):
    """parse_book + error handling + a polite pause; records failed URLs."""
    try:
        record = parse_book(url)
    except Exception as e:                      # e.g. an unexpected page layout
        log.error("Failed to parse %s: %s", url, e)
        record = None
    if record is None:
        failed.append(url)
    time.sleep(DELAY)
    return record

cache_ok = (os.path.exists(RAW_FILE) and not FORCE_RESCRAPE
            and "price_text" in pd.read_csv(RAW_FILE, nrows=1).columns)

if cache_ok:
    raw_df = pd.read_csv(RAW_FILE, dtype=str, encoding="utf-8-sig")
    print(f"Loaded {len(raw_df)} records from the saved {RAW_FILE} (no requests sent). "
          "Set FORCE_RESCRAPE = True to download again.")
else:
    start = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        results = list(pool.map(safe_parse, book_urls))   # map keeps the site's original order
    raw_df = pd.DataFrame([r for r in results if r is not None])
    print(f"Scraped {len(raw_df)} of {len(book_urls)} books in {time.time() - start:.1f} s")
    print("Failed pages:", len(failed), failed[:5])

Scraped 1000 of 1000 books in 91.9 s
Failed pages: 0 []


**What was done:** Visited all product pages using 5 parallel workers and collected one record per book. Any page that failed is logged and listed in `failed`.

**Why:**
- Downloading pages one by one would take several minutes; a small thread pool (5 workers) makes it several times faster while still keeping the load on the server low. Each worker also pauses for `DELAY` seconds after each request.
- `pool.map` returns the results **in the same order as the site**, so the dataset is reproducible.
- `safe_parse` catches any unexpected error, so one bad page cannot stop the whole run, and every failure is recorded so it can be reported instead of hidden.
- **Caching:** if `books_raw.csv` already exists, it is loaded instead of scraping again. This avoids sending 1,000 unnecessary requests every time the notebook is re-run, in line with the project's responsible-scraping instruction.

**What the output shows / observations:**
<!-- VERIFY -->
- **1000 of 1000** books were scraped, with **0 failed pages**. *(If you re-run with the cache, the message says the data was loaded from the CSV.)*

## 7. Raw dataset

In [7]:
if not cache_ok:
    raw_df.to_csv(RAW_FILE, index=False, encoding="utf-8-sig")
print("Saved:", RAW_FILE, "| shape:", raw_df.shape)
raw_df.head()

Saved: books_raw.csv | shape: (1000, 14)


,title,category,price_text,price_excl_tax_text,price_incl_tax_text,tax_text,rating_text,availability_text,num_reviews_text,upc,product_type,description,image_url,url
0,A Light in the Attic,Poetry,£51.77,£51.77,£51.77,£0.00,Three,In stock (22 available),0,a897fe39b1053632,Books,It's hard to imagine a world without A Light i...,https://books.toscrape.com/media/cache/fe/72/f...,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Historical Fiction,£53.74,£53.74,£53.74,£0.00,One,In stock (20 available),0,90fa61229261140a,Books,"""Erotic and absorbing...Written with starling ...",https://books.toscrape.com/media/cache/08/e9/0...,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Fiction,£50.10,£50.10,£50.10,£0.00,One,In stock (20 available),0,6957f44c3847a760,Books,"Dans une France assez proche de la nôtre, un h...",https://books.toscrape.com/media/cache/ee/cf/e...,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Mystery,£47.82,£47.82,£47.82,£0.00,Four,In stock (20 available),0,e00eb4fd7b871a48,Books,"WICKED above her hipbone, GIRL across her hear...",https://books.toscrape.com/media/cache/c0/59/c...,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,History,£54.23,£54.23,£54.23,£0.00,Five,In stock (20 available),0,4165285e1663650f,Books,From a renowned historian comes a groundbreaki...,https://books.toscrape.com/media/cache/ce/5f/c...,https://books.toscrape.com/catalogue/sapiens-a...


**What was done:** Saved the scraped text, unchanged, to `books_raw.csv` and displayed the first five rows.

**Why:** The deliverables ask for both a **raw collected dataset** and a **final structured dataset**. Keeping the untouched version means every later cleaning step can be checked against what the website actually showed. The file is saved with `utf-8-sig` encoding so that the `£` sign and special characters in titles display correctly when the file is opened in Excel.

**What the output shows / observations:**
<!-- VERIFY -->
- The raw dataset has **1000 rows × 14 columns**, one row per book.
- Prices still contain the `£` sign, the rating is a word and the availability is a sentence (`In stock (22 available)`), so the columns are text and cannot yet be used for calculations. This is fixed in the next step.

## 8. Structured dataset (converting text to proper types)

In [8]:
def to_number(series):
    """'£51.77' -> 51.77 ; 'In stock (22 available)' -> 22 ; missing stays missing."""
    return pd.to_numeric(series.str.extract(r"(\d+(?:\.\d+)?)")[0], errors="coerce")

avail = raw_df["availability_text"]

df = pd.DataFrame({
    "title":          raw_df["title"],
    "category":       raw_df["category"],
    "price":          to_number(raw_df["price_text"]),
    "price_excl_tax": to_number(raw_df["price_excl_tax_text"]),
    "price_incl_tax": to_number(raw_df["price_incl_tax_text"]),
    "tax":            to_number(raw_df["tax_text"]),
    "rating":         raw_df["rating_text"].map(RATING_MAP).astype("Int64"),
    "availability":   avail.str.replace(r"\s*\(.*\)", "", regex=True).str.strip(),   # 'In stock'
    "in_stock":       avail.str.contains("In stock").astype("boolean"),
    "stock":          to_number(avail).astype("Int64"),
    "num_reviews":    to_number(raw_df["num_reviews_text"]).astype("Int64"),
    "upc":            raw_df["upc"],
    "product_type":   raw_df["product_type"],
    "description":    (raw_df["description"]
                       .str.replace(r"\s*\.\.\.more$", "", regex=True)   # remove the site's '...more' link text
                       .str.strip()
                       .replace("", pd.NA)),
    "image_url":      raw_df["image_url"],
    "url":            raw_df["url"],
})

df.to_csv(STRUCTURED_FILE, index=False, encoding="utf-8-sig")
print("Saved:", STRUCTURED_FILE, "| shape:", df.shape)
df.info()

Saved: books_structured.csv | shape: (1000, 16)
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           1000 non-null   str    
 1   category        1000 non-null   str    
 2   price           1000 non-null   float64
 3   price_excl_tax  1000 non-null   float64
 4   price_incl_tax  1000 non-null   float64
 5   tax             1000 non-null   float64
 6   rating          1000 non-null   Int64  
 7   availability    1000 non-null   str    
 8   in_stock        1000 non-null   boolean
 9   stock           1000 non-null   Int64  
 10  num_reviews     1000 non-null   Int64  
 11  upc             1000 non-null   str    
 12  product_type    1000 non-null   str    
 13  description     998 non-null    str    
 14  image_url       1000 non-null   str    
 15  url             1000 non-null   str    
dtypes: Int64(3), boolean(1), float64(4), str(8

**What was done:** Built the structured dataset from the raw text and saved it as `books_structured.csv`:
- **Prices and tax:** the number is extracted from text such as `£51.77` using a regular expression and stored as a decimal (`float`).
- **Rating:** the word (`"Three"`) is mapped to an integer from 1 to 5.
- **Availability** is split into three columns: the status text (`availability`, e.g. *In stock*), a Boolean `in_stock`, and the number of copies (`stock`).
- **Number of reviews** is converted to an integer.
- **Description:** the trailing `...more` (website link text, not part of the description) is removed; empty descriptions become missing.

**Why:**
- Machine-learning models and statistics need numbers, not text such as `£51.77`.
- Missing values are kept as **missing** (`NaN`/`<NA>`) and never replaced by 0. A missing stock count is not the same as "0 in stock", and filling it with 0 would invent data.
- The pandas types `Int64` and `boolean` are used because, unlike the normal `int` and `bool` types, they can hold missing values.
- `in_stock` gives the later sections a ready **Boolean** variable, and `stock` is the stock information the project asks for.

**What the output shows / observations:**
<!-- VERIFY -->
- 1000 rows × 16 columns. Numerical: `price`, `price_excl_tax`, `price_incl_tax`, `tax`, `rating`, `stock`, `num_reviews`. Categorical: `category`, `availability`, `product_type`. Boolean: `in_stock`. Text / identifiers: `title`, `description`, `upc`, `image_url`, `url`.
- Every column has 1000 non-null values except `description`, which has a few missing (books with no description on the site).

## 9. Data-quality report
The project asks us to report the actual number of unique records and any fields that are unavailable or inconsistent.

In [9]:
print("=== Records ===")
print("Book links collected     :", len(book_urls))
print("Records scraped          :", len(df))
print("Unique UPCs              :", df["upc"].nunique())
print("Unique URLs              :", df["url"].nunique())
print("Exact duplicate rows     :", df.duplicated().sum())
print("Books sharing a title    :", df["title"].duplicated(keep=False).sum())

print("\n=== Missing values per column ===")
missing = pd.DataFrame({"missing": df.isna().sum(),
                        "percent": (df.isna().mean() * 100).round(2)})
print(missing[missing["missing"] > 0] if missing["missing"].any() else "No missing values")

print("\n=== Constant columns (only one value for every book) ===")
for col in df.columns:
    if df[col].nunique(dropna=True) <= 1:
        print(f"{col:15s} -> {df[col].dropna().iloc[0]}")

print("\n=== Repeated price fields ===")
print("price == price_incl_tax for every book       :", (df["price"] == df["price_incl_tax"]).all())
print("price_excl_tax == price_incl_tax for every book:", (df["price_excl_tax"] == df["price_incl_tax"]).all())

print("\n=== Other checks ===")
print("Raw descriptions ending in '...more':", raw_df["description"].str.endswith("...more").sum())
print("Rating values :", sorted(df["rating"].dropna().unique().tolist()))
print("Price range   : £%.2f to £%.2f" % (df["price"].min(), df["price"].max()))
print("Stock range   :", df["stock"].min(), "to", df["stock"].max())

print("\n=== Categories ===")
print("Categories in data   :", df["category"].nunique())
print("Categories in sidebar:", len(site_categories))
print("In sidebar but not in data:", set(site_categories) - set(df["category"]))
print("Placeholder-looking categories:")
print(df["category"].value_counts()[["Default", "Add a comment"]]
      if {"Default", "Add a comment"} <= set(df["category"]) else "none")

=== Records ===
Book links collected     : 1000
Records scraped          : 1000
Unique UPCs              : 1000
Unique URLs              : 1000
Exact duplicate rows     : 0
Books sharing a title    : 2

=== Missing values per column ===
             missing  percent
description        2      0.2

=== Constant columns (only one value for every book) ===
tax             -> 0.0
availability    -> In stock
in_stock        -> True
num_reviews     -> 0
product_type    -> Books

=== Repeated price fields ===
price == price_incl_tax for every book       : True
price_excl_tax == price_incl_tax for every book: True

=== Other checks ===
Raw descriptions ending in '...more': 943
Rating values : [1, 2, 3, 4, 5]
Price range   : £10.00 to £59.99
Stock range   : 1 to 22

=== Categories ===
Categories in data   : 50
Categories in sidebar: 50
In sidebar but not in data: set()
Placeholder-looking categories:
category
Default          152
Add a comment     67
Name: count, dtype: int64


**What was done:** Checked the dataset for completeness and consistency: number of unique records, duplicates, missing values, columns that never change, repeated price fields, leftover website text, value ranges and categories.

**Why:** The project requires us to report the actual number of unique records (and not to fabricate or duplicate records), and to report any fields that are unavailable or inconsistent. These findings also tell the later sections which columns are useful and which must be dropped.

**What the output shows / observations:**
<!-- VERIFY: replace these with the exact numbers printed above -->
- **Unique records:** 1000 records, 1000 unique UPCs and 1000 unique URLs, with **no exact duplicate rows**. The dataset contains every book on the site exactly once. *(If some books share a title, they are still different products with different UPCs, so they are not duplicates.)*
- **Missing values:** only `description` is missing, for a small number of books that have no description on the site. All other fields are complete.
- **Constant fields:** `tax` is always £0.00, `num_reviews` is always 0, `product_type` is always "Books" and `availability` / `in_stock` are the same for every book (all books are in stock). Because they never change, they carry **no information** for modelling and should be dropped later (Section 10 of the brief asks us to check exactly this).
- **Repeated price fields:** `price`, `price_excl_tax` and `price_incl_tax` are identical for every book because tax is zero. If price is the regression target, the other two price columns and `tax` must be **excluded from the inputs**, since they would reveal the target.
- **Inconsistent text:** the raw descriptions end with `...more`, which comes from the website's layout; it was removed in the structured dataset.
- **Categories:** the data contains all 50 sidebar categories. Two of them, **"Default"** and **"Add a comment"**, look like placeholder labels rather than real genres, so they should be treated with care in the analysis.
- **Ranges:** ratings are 1 to 5, prices are roughly £10 to £60 and stock counts range from 1 to 22, all plausible values.

## 10. Summary and data dictionary

**Result:** all **1000 unique books** were collected from 50 catalogue pages and 1000 product pages, with no failed pages and no duplicates. <!-- VERIFY -->

**Files produced**

| File | Contents |
|---|---|
| `books_raw.csv` | Text exactly as scraped from the website (14 columns) |
| `books_structured.csv` | Typed, cleaned version, one book per row (16 columns): **the starting point for the rest of the project** |

**Data dictionary (`books_structured.csv`)**

| Column | Type | Description | Note for later sections |
|---|---|---|---|
| `title` | text | Book title | identifier, not a model input |
| `category` | categorical | Genre from the breadcrumb (50 values) | one-hot or target encoding; includes "Default" and "Add a comment" |
| `price` | float (£) | Displayed price | possible regression target |
| `price_excl_tax`, `price_incl_tax` | float (£) | Prices from the product table | identical to `price`, so drop if price is the target |
| `tax` | float (£) | Tax amount | always 0, so drop |
| `rating` | int 1–5 | Star rating | basis of the classification target; exclude from classification inputs |
| `availability` | categorical | Stock status text | constant, so drop |
| `in_stock` | Boolean | Whether the book is in stock | constant, so drop |
| `stock` | int | Number of copies available | usable numeric feature |
| `num_reviews` | int | Number of reviews | always 0, so drop |
| `upc` | text | Unique product code | identifier, do not use as a feature |
| `product_type` | categorical | Product type | always "Books", so drop |
| `description` | text | Product description | a few missing; could be used for TF-IDF (advanced technique) |
| `image_url`, `url` | text | Cover image and product page links | identifiers, not features |

**Limitations:** the site's values are generated for scraping practice, so several fields are constant and relationships between attributes (for example, price and rating) may be weak. The data reflects the site on the day it was scraped.